# Step 1 : Reading and Parsing the WhatsApp Chat

In this section we:

- Read the exported WhatsApp chat
- Handle multi-line messages
- Ignore system messages
- Count deleted/media messages
- Store every valid message in a structured format

In [1]:
import numpy as np
from datetime import datetime

In [2]:
FILE_PATH = "../Dataset/hostel_bois.txt"

with open(FILE_PATH, "r", encoding="utf-8") as file:
    lines = file.readlines()

print("Total Lines :", len(lines))

Total Lines : 83743


In [3]:
messages = []

system_messages = 0
media_messages = 0
deleted_messages = 0

current_message = ""

for line in lines:

    line = line.strip()

    if line == "":
        continue

    # Detect start of a new message
    if len(line) >= 10 and line[2] == "/" and line[5] == "/":

        if current_message != "":
            messages.append(current_message)

        current_message = line

    else:
        # Multi-line continuation
        current_message += " " + line

if current_message:
    messages.append(current_message)

print("Messages after merging multiline :", len(messages))

Messages after merging multiline : 79069


In [4]:
chat_data = []

for msg in messages:

    if " - " not in msg:
        continue

    timestamp, rest = msg.split(" - ", 1)

    # System message
    if ": " not in rest:
        system_messages += 1
        continue

    sender, text = rest.split(": ", 1)

    if text == "<Media omitted>":
        media_messages += 1

    if text == "This message was deleted":
        deleted_messages += 1

    clean_time = (
        timestamp
        .replace("\u202f", " ")
        .replace("am", "AM")
        .replace("pm", "PM")
    )

    dt = datetime.strptime(clean_time, "%d/%m/%Y, %I:%M %p")

    chat_data.append({
        "timestamp": dt,
        "sender": sender,
        "text": text
    })

In [5]:
print("="*60)

print("VALID MESSAGES :", len(chat_data))
print("SYSTEM :", system_messages)
print("MEDIA :", media_messages)
print("DELETED :", deleted_messages)

print("="*60)

print(chat_data[:5])

VALID MESSAGES : 78088
SYSTEM : 981
MEDIA : 9702
DELETED : 680
[{'timestamp': datetime.datetime(2023, 9, 4, 14, 29), 'sender': 'Anshley CSE A', 'text': 'Case kheygelam bro💔'}, {'timestamp': datetime.datetime(2023, 9, 4, 14, 29), 'sender': 'Anshley CSE A', 'text': '<Media omitted>'}, {'timestamp': datetime.datetime(2023, 9, 4, 14, 42), 'sender': 'Kaninika CSE', 'text': 'LMAO'}, {'timestamp': datetime.datetime(2023, 9, 4, 14, 44), 'sender': '+91 72788 80557', 'text': 'DR. Banhishikha Bhattacharya (Life Skills) 9433912075 counsellor@heritageit.edu CME-413'}, {'timestamp': datetime.datetime(2023, 9, 4, 14, 44), 'sender': 'Kaninika CSE', 'text': 'Are we supposed to talk to her'}]


In [6]:
# ===========================
# FEATURE 1 OUTPUT
# ===========================

print("=" * 70)
print("FEATURE 1 : CHAT PARSER")
print("=" * 70)

print(f"{'Total Lines in File':<35}: {len(lines):>10,}")
print(f"{'Merged Messages':<35}: {len(messages):>10,}")
print(f"{'Valid Messages':<35}: {len(chat_data):>10,}")
print(f"{'System Messages':<35}: {system_messages:>10,}")
print(f"{'Media Messages':<35}: {media_messages:>10,}")
print(f"{'Deleted Messages':<35}: {deleted_messages:>10,}")

print("=" * 70)

print("\nFIRST 5 PARSED MESSAGES\n")

for i, msg in enumerate(chat_data[:5], start=1):
    print(f"Message {i}")
    print(f"Timestamp : {msg['timestamp']}")
    print(f"Sender    : {msg['sender']}")
    print(f"Message   : {msg['text']}")
    print("-" * 70)

print("\nLAST 5 PARSED MESSAGES\n")

for i, msg in enumerate(chat_data[-5:], start=len(chat_data)-4):
    print(f"Message {i}")
    print(f"Timestamp : {msg['timestamp']}")
    print(f"Sender    : {msg['sender']}")
    print(f"Message   : {msg['text']}")
    print("-" * 70)

print("\nParser Status : SUCCESS")
print("=" * 70)

FEATURE 1 : CHAT PARSER
Total Lines in File                :     83,743
Merged Messages                    :     79,069
Valid Messages                     :     78,088
System Messages                    :        981
Media Messages                     :      9,702
Deleted Messages                   :        680

FIRST 5 PARSED MESSAGES

Message 1
Timestamp : 2023-09-04 14:29:00
Sender    : Anshley CSE A
Message   : Case kheygelam bro💔
----------------------------------------------------------------------
Message 2
Timestamp : 2023-09-04 14:29:00
Sender    : Anshley CSE A
Message   : <Media omitted>
----------------------------------------------------------------------
Message 3
Timestamp : 2023-09-04 14:42:00
Sender    : Kaninika CSE
Message   : LMAO
----------------------------------------------------------------------
Message 4
Timestamp : 2023-09-04 14:44:00
Sender    : +91 72788 80557
Message   : DR. Banhishikha Bhattacharya (Life Skills) 9433912075 counsellor@heritageit.edu CME-413

In [7]:
print("\nVALIDATION")

assert len(chat_data) > 0, "No messages parsed."

assert len(messages) >= len(chat_data), \
       "Merged messages cannot be fewer than parsed messages."

assert system_messages >= 0
assert media_messages >= 0
assert deleted_messages >= 0

assert isinstance(chat_data[0]["timestamp"], datetime)

print("All validation checks passed.")


VALIDATION
All validation checks passed.


In [9]:
import pickle

with open("../Data/parsed_chat.pkl", "wb") as file:
    pickle.dump(chat_data, file)

print("Parsed chat exported successfully.")

Parsed chat exported successfully.
